# Model Evaluation: Base Model vs. Fine-tuned BamiBERT

This notebook evaluates **Precision**, **Recall**, **F1-score**, and **Accuracy** on the NLI dataset:
- **Premise**: `specific_question`
- **Hypothesis**: `legal_document`
- **Label Mapping**: Trong file `vlsp_nli.parquet`, `choices = ['Có', 'Không']`, do đó `answer = 0` tương ứng với `'Có'` (Entailment = 1), và `answer = 1` tương ứng với `'Không'` (Non-entailment = 0).

In [1]:
import os
import json
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download
from transformers import PreTrainedTokenizerFast, AutoModelForSequenceClassification
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report

c:\Users\VINH\OneDrive - VNU-HCMUS\Attachments\Desktop\rag-textmining\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configurations
finetuned_model_path = "bamibert_vilegalnli_best"
base_model_path = "Qualcomm-AI-Research/BamiBERT"
dataset_path = "vlsp_nli.parquet"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Load test dataset
df = pd.read_parquet(dataset_path)

# Ánh xạ nhãn: answer 0 (lựa chọn 'Có') -> 1 (Entailment), answer 1 (lựa chọn 'Không') -> 0 (Non-entailment)
if 'choices' in df.columns:
    # choices = ['Có', 'Không'] -> 0: Có (Entailment = 1), 1: Không (Non-entailment = 0)
    df['nli_label'] = df['answer'].apply(lambda x: 1 if x == 0 else 0)
else:
    df['nli_label'] = df['answer']

label_col = 'nli_label'

print(f"Total test samples: {len(df)}")
print("\nLabel distribution (1: Entailment, 0: Non-entailment):")
print(df[label_col].value_counts())
print("\nSample data:")
print(df[['specific_question', 'legal_document', 'answer', label_col]].head())

FileNotFoundError: [Errno 2] No such file or directory: 'vlsp_nli.parquet'

In [ ]:
def get_tokenizer(model_name_or_path):
    local_tok_path = os.path.join(model_name_or_path, "tokenizer.json")
    if os.path.exists(local_tok_path):
        tok_file = local_tok_path
    else:
        tok_file = hf_hub_download(repo_id=model_name_or_path, filename="tokenizer.json")
    
    raw_tok = Tokenizer.from_file(tok_file)
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=raw_tok,
        bos_token="<s>",
        eos_token="</s>",
        unk_token="<unk>",
        sep_token="</s>",
        cls_token="<s>",
        pad_token="<pad>",
        mask_token="<mask model_max_length=2048>",
        model_max_length=2048
    )
    return tokenizer

def evaluate_model(model_name_or_path, df, batch_size=16):
    print(f"\n>>> Loading model and tokenizer from: {model_name_or_path}")
    tokenizer = get_tokenizer(model_name_or_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path,
        num_labels=2,
        ignore_mismatched_sizes=True
    ).to(device)
    model.eval()
    
    predictions = []
    
    for i in tqdm(range(0, len(df), batch_size), desc=f"Evaluating {os.path.basename(model_name_or_path)}"):
        batch = df.iloc[i:i+batch_size]
        
        premises = batch['specific_question'].tolist()
        hypotheses = batch['legal_document'].tolist()
        
        inputs = tokenizer(
            premises,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=2048,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        predictions.extend(preds)
        
    return predictions

In [ ]:
print("=== EVALUATING BASE MODEL ===")
base_preds = evaluate_model(base_model_path, df)

print("\n=== EVALUATING FINE-TUNED MODEL ===")
finetuned_preds = evaluate_model(finetuned_model_path, df)

=== EVALUATING BASE MODEL ===

>>> Loading model and tokenizer from: Qualcomm-AI-Research/BamiBERT


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3840.62it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: Qualcomm-AI-Research/BamiBERT
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.decoder.bias       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Evaluating BamiBERT: 100%|██████████| 10/10 [00:02<00:00,  3.67it/s]



=== EVALUATING FINE-TUNED MODEL ===

>>> Loading model and tokenizer from: bamibert_vilegalnli_best


Evaluating bamibert_vilegalnli_best: 100%|██████████| 10/10 [00:02<00:00,  4.03it/s]


In [ ]:
def calculate_metrics(y_true, y_pred):
    return {
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "F1-Score": round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "Macro F1": round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4)
    }

y_true = df[label_col].tolist()

base_metrics = calculate_metrics(y_true, base_preds)
finetuned_metrics = calculate_metrics(y_true, finetuned_preds)

comparison_df = pd.DataFrame([
    {"Model": "Base Model (BamiBERT)", **base_metrics},
    {"Model": "Fine-tuned Model (vilegalnli)", **finetuned_metrics}
])

print("\n" + "="*60)
print("                   COMPARISON SUMMARY")
print("="*60)
print(comparison_df.to_string(index=False))

print("\n" + "-"*60)
print("Detailed Classification Report: Base Model")
print("-"*60)
print(classification_report(y_true, base_preds, target_names=["Non-entailment (0)", "Entailment (1)"], zero_division=0))

print("-"*60)
print("Detailed Classification Report: Fine-tuned Model")
print("-"*60)
print(classification_report(y_true, finetuned_preds, target_names=["Non-entailment (0)", "Entailment (1)"], zero_division=0))


                   COMPARISON SUMMARY
                        Model  Accuracy  Precision  Recall  F1-Score  Macro F1
        Base Model (BamiBERT)      0.50     0.5000  1.0000    0.6667    0.3333
Fine-tuned Model (vilegalnli)      0.76     0.7349  0.8133    0.7722    0.7593

------------------------------------------------------------
Detailed Classification Report: Base Model
------------------------------------------------------------
                    precision    recall  f1-score   support

Non-entailment (0)       0.00      0.00      0.00        75
    Entailment (1)       0.50      1.00      0.67        75

          accuracy                           0.50       150
         macro avg       0.25      0.50      0.33       150
      weighted avg       0.25      0.50      0.33       150

------------------------------------------------------------
Detailed Classification Report: Fine-tuned Model
------------------------------------------------------------
                    prec